# The polynomial Stokes problem of Araya et al. (2017)

We use the exact velocity and pressure from §3.1.1 of [the 2017 paper](https://doi.org/10.1016/j.cma.2017.05.027): a streamfunction $\psi=-128x^2(x-1)^2y^2(y-1)^2$, $u=(\psi_y,-\psi_x)$, and $p=150(x-1/2)(y-1/2)$. Sources are differentiated from these fields. These diagonal triangulations and P2/P1 or P1/P1 local spaces differ from the paper's complete convergence experiments. Assembly uses Duffy order 5 and error integration uses order 8, sufficient for the respective polynomial degrees.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace, solve_darcy, solve_brinkman, solve_elasticity


In [ ]:
from numpy.polynomial import Polynomial
a=Polynomial([0,0,1,-2,1])
def velocity(x):
    """Differentiate the streamfunction to obtain exactly solenoidal velocity."""
    X,Y=x.T
    return 128*np.column_stack((-a(X)*a.deriv()(Y),a.deriv()(X)*a(Y)))
def pressure(x):
    """Return the analytical zero-mean pressure."""
    return 150*(x[:,0]-.5)*(x[:,1]-.5)
def source(x):
    """Return negative velocity Laplacian plus the pressure gradient."""
    X,Y=x.T
    lap=128*np.column_stack((-(a.deriv(2)(X)*a.deriv()(Y)+a(X)*a.deriv(3)(Y)),a.deriv(3)(X)*a(Y)+a.deriv()(X)*a.deriv(2)(Y)))
    return -lap+150*np.column_stack((Y-.5,X-.5))
for formulation in ('taylor-hood','usfem'):
    errors=[]
    for n in (2,4):
        mesh=TriangleMesh.unit_square(n)
        space=SkeletonSpace(mesh,tuple(FaceSpace.uniform(1) for _ in mesh.faces),2)
        result=solve_brinkman(mesh,source=source,dirichlet=velocity,skeleton=space,local_refinement=4,formulation=formulation)
        errors.append(result.l2_error(velocity,order=8))
        print(formulation,n,'velocity',errors[-1],'pressure',result.pressure_l2_error(pressure,order=8),'divergence',result.divergence_l2())
    assert errors[1]<errors[0]/3


The discrete velocity is weakly incompressible and need not have pointwise zero divergence. A global pressure gauge is essential. Residual convergence and a divergence-free analytical velocity do not alone prove pointwise discrete incompressibility.